# C3 round 3 — 5-seed confirmation of the two best round-2 candidates
Round 2 (seed 0) ranking on the near-duplicate-free split: **DINOv2 ViT-S/14 + linear head** (macro-F1 94.12;
glaze 86.0, frost 92.0, snow 90.9) and **Swin-T + KAN head + CB-focal, head LR 1×** (94.07). Swin-T baseline: 93.37.
This notebook runs seeds 1–4 of both (seed 0 is reused), and also saves validation probabilities for calibration.

**Before running:** GPU T4 · **Internet On** (DINOv2 weights) · Add Input → Weather Image Recognition (jehanbhathena).
About 1.8 h. Results go to `/kaggle/working/results_c3r2/` and `results_c3r2_seeds1234_small.zip`.

In [ ]:
# 0. CONFIG — the only cell you normally edit
CFG = dict(
    RUN=["dino_lin_ce", "swin_kan_cbf_h1"],
    PROPOSED="dino_lin_ce",
    SEEDS=[1, 2, 3, 4],       # seed 0 of both experiments is already done (round 2); merged afterwards
    MAX_HOURS=11.3,
    KAN_HIDDEN=256, KAN_GRID=8, CB_BETA=0.999, FOCAL_GAMMA=2.0,
    AUX_W=0.5, ICE_CLASSES=["frost", "glaze", "hail", "rime", "snow"],
    PHASH_MAX_DIST=8, SPLIT_SEED=42, IMG_RESIZE=256, IMG_SIZE=224,
    BATCH=32, EPOCHS=30, WD=0.05, PATIENCE=8,
    SAVE_WEIGHTS_SEEDS=[0], NUM_WORKERS=4, DATA_ROOT=None, WORK="/kaggle/working",
)
SWIN, DINO, VIT384 = "swin_tiny_patch4_window7_224", "vit_small_patch14_dinov2.lvd142m", "vit_small_patch16_384.augreg_in21k_ft_in1k"
# exp: (backbone, head, loss, image size, backbone LR, head LR)
EXPS = {
    "swin_ice_ce":     (SWIN, "ice", "ce", 224, 1e-4, 1e-4),
    "dino_lin_ce":     (DINO, "linear", "ce", 224, 2e-5, 1e-3),
    "dino_ice_ce":     (DINO, "ice", "ce", 224, 2e-5, 1e-3),
    "vit384_lin_ce":   (VIT384, "linear", "ce", 384, 1e-4, 1e-4),
    "swin_kan_cbf_h1": (SWIN, "kan", "cbfocal", 224, 1e-4, 1e-4),
    "swin_lin_ce":     (SWIN, "linear", "ce", 224, 1e-4, 1e-4),
    "dino384_ice_ce":  (DINO, "ice", "ce", 392, 2e-5, 1e-3),
}
print(CFG)

In [ ]:
# 1. Imports and environment
import os, glob, json, time, random, hashlib, shutil, subprocess, sys
import numpy as np, pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True); import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix, classification_report)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T_START = time.time()
print("torch", torch.__version__, "| timm", timm.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
WORK = CFG["WORK"]; RES = os.path.join(WORK, "results_c3r2"); os.makedirs(RES, exist_ok=True)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Resume across commits: if an earlier version's output was added as an Input, copy its results here
# (same split.csv, finished runs are then skipped).
prev_runs = []
for prev in glob.glob("/kaggle/input/**/results_c3r2/runs.csv", recursive=True):
    for f in glob.glob(os.path.join(os.path.dirname(prev), "*")):
        dst = os.path.join(RES, os.path.basename(f))
        if os.path.basename(f) != "runs.csv" and not os.path.exists(dst): shutil.copy(f, dst)
    prev_runs.append(pd.read_csv(prev)); print("Resumed results from", os.path.dirname(prev))
if prev_runs:
    pd.concat(prev_runs).drop_duplicates(["model", "seed"], keep="last").to_csv(os.path.join(RES, "runs.csv"), index=False)

## 1. Find WEAPD
Two ways to give the notebook the data (Add Input on the right):
* **A. Original (recommended for the paper):** upload `dataset(WEAPD).rar` from Harvard Dataverse as a private
  Kaggle dataset. Kaggle keeps .rar files as they are, so this cell extracts it.
* **B. Kaggle copy:** search "Weather Image Recognition" by jehanbhathena (6,862 images). Quicker, but say so in the paper.

In [ ]:
def find_class_root(base):
    """Return the folder whose sub-folders are the 11 classes (most image-holding sub-folders)."""
    best, best_n = None, 0
    for d, subdirs, _ in os.walk(base):
        n = sum(1 for s in subdirs
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(d, s))[:50]))
        if n > best_n:
            best, best_n = d, n
    return best if best_n >= 5 else None

def extract_archives(src, dst):
    os.makedirs(dst, exist_ok=True)
    for a in glob.glob(os.path.join(src, "**", "*.rar"), recursive=True) + \
             glob.glob(os.path.join(src, "**", "*.zip"), recursive=True):
        print("Extracting", a)
        if a.endswith(".zip"):
            shutil.unpack_archive(a, dst)
        else:
            ok = subprocess.run(["7z", "x", "-y", f"-o{dst}", a], capture_output=True).returncode == 0
            if not ok:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rarfile", "unrar"], check=False)
                subprocess.run(["bash", "-c", "apt-get -qq install -y unrar >/dev/null 2>&1 || true"])
                subprocess.run(["unrar", "x", "-o+", a, dst + "/"], check=True)

DATA_ROOT = CFG["DATA_ROOT"] or find_class_root("/kaggle/input")
if DATA_ROOT is None:
    extract_archives("/kaggle/input", os.path.join(WORK, "raw"))
    DATA_ROOT = find_class_root(os.path.join(WORK, "raw"))
assert DATA_ROOT, "WEAPD not found: add the dataset as an Input (Section 1)."
CLASSES = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
print("DATA_ROOT:", DATA_ROOT); print(len(CLASSES), "classes:", CLASSES)

## 2. Clean, remove exact and near-duplicates, resize once, and fix the split
Every readable image gets an MD5 and a 64-bit perceptual hash (DCT of a 32×32 grey image, top-left 8×8, median
threshold). Images with the same MD5 or a pHash Hamming distance <= `PHASH_MAX_DIST` are joined into one group
(union–find). Groups with more than one label are dropped; from every other group the largest image is kept.
Kept images are resized to 256×256 once and cached.

In [ ]:
CACHE = os.path.join(WORK, f"cache{CFG['IMG_RESIZE']}")
split_csv = os.path.join(RES, "split.csv")
_N = 32
_C = np.array([[np.cos(np.pi * (2 * x + 1) * u / (2 * _N)) for x in range(_N)] for u in range(_N)])
_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def phash64(im):
    a = np.asarray(im.convert("L").resize((_N, _N), Image.LANCZOS), dtype=np.float64)
    low = (_C @ a @ _C.T)[:8, :8].flatten()
    return format(int("".join("1" if v else "0" for v in low > np.median(low[1:])), 2), "016x")

def hamming(code, codes):
    return _POP[(codes ^ np.uint64(code)).view(np.uint8).reshape(-1, 8)].sum(1)

if not os.path.exists(split_csv):
    rows, n_bad = [], 0
    for c in CLASSES:
        for f in sorted(os.listdir(os.path.join(DATA_ROOT, c))):
            if not f.lower().endswith(IMG_EXT): continue
            p = os.path.join(DATA_ROOT, c, f)
            try:
                raw = open(p, "rb").read(); im = Image.open(p); im.load(); im = im.convert("RGB")
            except Exception:
                n_bad += 1; continue
            rows.append(dict(src=p, label=c, file=f, md5=hashlib.md5(raw).hexdigest(), phash=phash64(im),
                             w=im.size[0], h=im.size[1]))
    al = pd.DataFrame(rows); n = len(al)
    codes = np.array([int(x, 16) for x in al.phash], dtype=np.uint64)
    par = list(range(n))
    def find(i):
        while par[i] != i: par[i] = par[par[i]]; i = par[i]
        return i
    first = {}
    for i, m in enumerate(al.md5):                       # exact duplicates
        if m in first: par[find(i)] = find(first[m])
        else: first[m] = i
    TH = CFG["PHASH_MAX_DIST"]
    for i in range(n - 1):                               # near-duplicates
        for j in np.nonzero(hamming(codes[i], codes[i + 1:]) <= TH)[0]:
            par[find(i)] = find(i + 1 + int(j))
    al["group"] = [find(i) for i in range(n)]
    gsize = al.group.map(al.group.value_counts()); glabels = al.group.map(al.groupby("group").label.nunique())
    al["exact_dup"] = al.duplicated("md5")
    al["conflict"] = glabels > 1
    al["area"] = al.w * al.h
    keep_idx = al[~al.conflict].sort_values(["group", "area"], ascending=[True, False]).drop_duplicates("group").index
    al["kept"] = al.index.isin(keep_idx)
    al["reason"] = np.where(al.kept, "kept", np.where(al.conflict, "label_conflict",
                            np.where(al.exact_dup, "exact_duplicate", "near_duplicate")))
    audit = al.pivot_table(index="label", columns="reason", values="file", aggfunc="count", fill_value=0)
    audit["original"] = audit.sum(axis=1); audit.loc["TOTAL"] = audit.sum()
    audit.to_csv(os.path.join(RES, "dedup_audit.csv")); al.drop(columns=["area"]).to_csv(os.path.join(RES, "all_images_hashes.csv"), index=False)
    print(f"{n} readable images ({n_bad} unreadable) -> {al.kept.sum()} kept | "
          f"{(al.group.value_counts() > 1).sum()} duplicate groups | {al.conflict.sum()} images in label-conflict groups")
    df = al[al.kept].copy()
    for c in CLASSES: os.makedirs(os.path.join(CACHE, c), exist_ok=True)
    df["path"] = [os.path.join(CACHE, r.label, os.path.splitext(r.file)[0] + ".jpg") for r in df.itertuples()]
    for r in df.itertuples():
        Image.open(r.src).convert("RGB").resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC).save(r.path, quality=95)
    df = df[["path", "label", "src", "md5", "phash", "group"]].reset_index(drop=True)
    tr, tmp = train_test_split(df, test_size=0.2, stratify=df.label, random_state=CFG["SPLIT_SEED"])
    va, te = train_test_split(tmp, test_size=0.5, stratify=tmp.label, random_state=CFG["SPLIT_SEED"])
    df["split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"
    df.to_csv(split_csv, index=False)
    print(audit)
df = pd.read_csv(split_csv, dtype={"phash": str})
if not os.path.exists(df.path.iloc[0]):   # cache lost after a restart: rebuild from src
    for r in df.itertuples():
        os.makedirs(os.path.dirname(r.path), exist_ok=True)
        Image.open(r.src).convert("RGB").resize((CFG["IMG_RESIZE"],) * 2, Image.BICUBIC).save(r.path, quality=95)
# sanity check: no test/val image within the threshold of any train image
_tr = np.array([int(x, 16) for x in df[df.split == "train"].phash], dtype=np.uint64)
_close = sum(int(hamming(int(x, 16), _tr).min() <= CFG["PHASH_MAX_DIST"]) for x in df[df.split != "train"].phash)
print("val/test images with a near-duplicate in train:", _close)
C2I = {c: i for i, c in enumerate(CLASSES)}
counts = df.pivot_table(index="label", columns="split", values="path", aggfunc="count")[["train", "val", "test"]]
counts.loc["TOTAL"] = counts.sum(); counts.to_csv(os.path.join(RES, "class_counts.csv")); counts

In [ ]:
# 3. Data loaders (Xiao et al. 2021 augmentation: random crop, flip, rotation 15°, random erasing).
# Images are cached once per resolution: 256 for 224-px models, round(size·256/224) for larger inputs.
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def cached_paths(size):
    if size == CFG["IMG_SIZE"]: return df.path.tolist()
    res = int(round(size * CFG["IMG_RESIZE"] / CFG["IMG_SIZE"])); root = os.path.join(WORK, f"cache{res}")
    out = []
    for r in df.itertuples():
        p = os.path.join(root, r.label, os.path.basename(r.path))
        if not os.path.exists(p):
            os.makedirs(os.path.dirname(p), exist_ok=True)
            Image.open(r.src).convert("RGB").resize((res, res), Image.BICUBIC).save(p, quality=95)
        out.append(p)
    return out

class ImgDS(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = paths, [C2I[l] for l in labels], tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

def loaders(batch, seed, size=224):
    paths = np.array(cached_paths(size))
    tf_tr = T.Compose([T.RandomCrop(size), T.RandomHorizontalFlip(), T.RandomRotation(15),
                       T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
    tf_ev = T.Compose([T.CenterCrop(size), T.ToTensor(), T.Normalize(MEAN, STD)])
    g = torch.Generator().manual_seed(seed)
    kw = dict(num_workers=CFG["NUM_WORKERS"], pin_memory=DEVICE == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0)
    mk = lambda sp: (list(paths[(df.split == sp).values]), df[df.split == sp].label.tolist())
    return (DataLoader(ImgDS(*mk("train"), tf_tr), batch, shuffle=True, drop_last=True, generator=g, **kw),
            DataLoader(ImgDS(*mk("val"), tf_ev), 64, **kw),
            DataLoader(ImgDS(*mk("test"), tf_ev), 64, **kw))

## 4. Models and losses

In [ ]:
class RBFKANLayer(nn.Module):
    """FastKAN-style layer: y = W_s · φ(LN(x)) + W_b · SiLU(x), φ = Gaussian RBFs on a fixed grid."""
    def __init__(self, din, dout, grid=8, lo=-2.0, hi=2.0):
        super().__init__()
        self.norm = nn.LayerNorm(din)
        self.register_buffer("centres", torch.linspace(lo, hi, grid))
        self.inv_h = (grid - 1) / (hi - lo)
        self.spline = nn.Linear(din * grid, dout); self.base = nn.Linear(din, dout)
        nn.init.trunc_normal_(self.spline.weight, std=0.02); nn.init.zeros_(self.spline.bias)
    def forward(self, x):
        z = self.norm(x).float()
        phi = torch.exp(-((z[..., None] - self.centres) * self.inv_h) ** 2).flatten(1)
        return self.spline(phi.to(x.dtype)) + self.base(F.silu(x))

ICE_IDX = [C2I[c] for c in CFG["ICE_CLASSES"]]
ICE_MAP = torch.full((len(CLASSES),), -1, dtype=torch.long)
for j, i in enumerate(ICE_IDX): ICE_MAP[i] = j

class Net(nn.Module):
    def __init__(self, backbone, head, n_classes, size):
        super().__init__()
        kw = dict(pretrained=True, num_classes=0)
        if "dinov2" in backbone: kw["img_size"] = size
        self.backbone = timm.create_model(backbone, **kw)
        d = self.backbone.num_features; self.kind = head
        if head == "kan":
            self.head = nn.Sequential(RBFKANLayer(d, CFG["KAN_HIDDEN"], CFG["KAN_GRID"]), nn.Dropout(0.1),
                                      RBFKANLayer(CFG["KAN_HIDDEN"], n_classes, CFG["KAN_GRID"]))
        else:
            self.head = nn.Linear(d, n_classes)
        if head == "ice":
            self.group = nn.Linear(d, 2)
            self.expert = nn.Sequential(nn.Linear(d, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, len(ICE_IDX)))
            self.register_buffer("ice_idx", torch.tensor(ICE_IDX))
    def head_parameters(self):
        mods = [self.head] + ([self.group, self.expert] if self.kind == "ice" else [])
        return [p for m in mods for p in m.parameters()]
    def forward(self, x): return self.head(self.backbone(x))
    def outputs(self, x):
        f = self.backbone(x); o = {"main": self.head(f)}
        if self.kind == "ice": o["group"] = self.group(f); o["expert"] = self.expert(f)
        return o
    def probs(self, x):
        o = self.outputs(x); p = o["main"].float().softmax(1)
        if self.kind == "ice":
            pe = o["expert"].float().softmax(1); pi = p[:, self.ice_idx]
            p = p.clone(); p[:, self.ice_idx] = 0.5 * pi + 0.5 * pi.sum(1, keepdim=True) * pe
        return p

class CBFocalLoss(nn.Module):
    def __init__(self, counts, beta=0.999, gamma=2.0):
        super().__init__()
        eff = 1.0 - np.power(beta, np.asarray(counts, dtype=np.float64))
        w = (1.0 - beta) / eff; w = w / w.sum() * len(counts)
        self.register_buffer("w", torch.tensor(w, dtype=torch.float32)); self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), 1).gather(1, y[:, None]).squeeze(1)
        return (self.w[y] * (1 - logp.exp()) ** self.gamma * -logp).mean()

def complexity(m, size):
    params = sum(p.numel() for p in m.parameters()) / 1e6
    try:
        from torch.utils.flop_counter import FlopCounterMode
        m.eval(); x = torch.randn(1, 3, size, size, device=next(m.parameters()).device)
        with FlopCounterMode(display=False) as fc: m(x)
        gflops = fc.get_total_flops() / 2e9
    except Exception:
        gflops = float("nan")
    return params, gflops

TRAIN_COUNTS = [int((df[df.split == "train"].label == c).sum()) for c in CLASSES]
print("train counts:", dict(zip(CLASSES, TRAIN_COUNTS)))

In [ ]:
# 5. Train / evaluate one (experiment, seed) run
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True

@torch.no_grad()
def predict(m, dl, tta=False):
    m.eval(); P, Y = [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            p = m.probs(x)
            if tta: p = (p + m.probs(torch.flip(x, dims=[3]))) / 2
        P.append(p.cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return dict(acc=accuracy_score(y, p), macro_p=pr, macro_r=rc, macro_f1=f1,
                weighted_f1=f1_score(y, p, average="weighted"), kappa=cohen_kappa_score(y, p), mcc=matthews_corrcoef(y, p))

def run(exp, seed):
    backbone, head, loss, size, lr_b, lr_h = EXPS[exp]
    set_seed(seed)
    dl_tr, dl_va, dl_te = loaders(CFG["BATCH"], seed, size)
    m = Net(backbone, head, len(CLASSES), size).to(DEVICE)
    hp = m.head_parameters(); hid = {id(p) for p in hp}
    bp = [p for p in m.parameters() if id(p) not in hid]
    opt = torch.optim.AdamW([{"params": bp, "lr": lr_b}, {"params": hp, "lr": lr_h}], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr_b, lr_h], total_steps=CFG["EPOCHS"] * len(dl_tr), pct_start=0.1)
    ice_map = ICE_MAP.to(DEVICE)
    crit = (CBFocalLoss(TRAIN_COUNTS, CFG["CB_BETA"], CFG["FOCAL_GAMMA"]) if loss == "cbfocal" else nn.CrossEntropyLoss()).to(DEVICE)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_ep, wait, hist, t0 = -1, 0, 0, [], time.time()
    ckpt = os.path.join(WORK, f"best_{exp}_s{seed}.pt")
    for ep in range(1, CFG["EPOCHS"] + 1):
        m.train(); tl, n = 0.0, 0
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                o = m.outputs(x); l = crit(o["main"], y)
                if head == "ice":
                    yi = ice_map[y]
                    l = l + CFG["AUX_W"] * F.cross_entropy(o["group"].float(), (yi >= 0).long())
                    if (yi >= 0).any():
                        l = l + CFG["AUX_W"] * F.cross_entropy(o["expert"].float()[yi >= 0], yi[yi >= 0])
            scaler.scale(l).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += l.item() * len(y); n += len(y)
        pv, yv = predict(m, dl_va)
        vf1 = f1_score(yv, pv.argmax(1), average="macro")
        hist.append(dict(epoch=ep, train_loss=tl / n, val_acc=accuracy_score(yv, pv.argmax(1)), val_macro_f1=vf1))
        if vf1 > best: best, best_ep, wait = vf1, ep, 0; torch.save(m.state_dict(), ckpt)
        else: wait += 1
        print(f"  {exp} s{seed} ep{ep:2d} loss {tl/n:.4f} val_F1 {vf1:.4f}{' *' if wait == 0 else ''}")
        if wait >= CFG["PATIENCE"]: break
    train_min = (time.time() - t0) / 60
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    pt, yt = predict(m, dl_te); ptt, _ = predict(m, dl_te, tta=True)
    pvb, yvb = predict(m, dl_va)                      # validation probabilities of the best checkpoint (for calibration)
    np.save(os.path.join(RES, f"valprob_{exp}_s{seed}.npy"), pvb); np.save(os.path.join(RES, "val_labels.npy"), yvb)
    params, gflops = complexity(m, size)
    m.eval(); x1 = torch.randn(1, 3, size, size, device=DEVICE)
    with torch.no_grad():
        for _ in range(10): m(x1)
        if DEVICE == "cuda": torch.cuda.synchronize()
        t1 = time.time()
        for _ in range(50): m(x1)
        if DEVICE == "cuda": torch.cuda.synchronize()
    ms = (time.time() - t1) / 50 * 1000
    yp = pt.argmax(1); tag = f"{exp}_s{seed}"
    out = dict(exp=exp, backbone=backbone, head=head, loss=loss, img_size=size, seed=seed, best_epoch=best_ep, epochs_run=len(hist),
               train_min=round(train_min, 1), params_M=round(params, 2), gflops=round(gflops, 2), ms_per_img=round(ms, 2),
               **metrics(yt, yp), acc_tta=accuracy_score(yt, ptt.argmax(1)), macro_f1_tta=f1_score(yt, ptt.argmax(1), average="macro"))
    rep = classification_report(yt, yp, labels=list(range(len(CLASSES))), target_names=CLASSES, output_dict=True, zero_division=0)
    for c in CLASSES: out[f"f1_{c}"] = rep[c]["f1-score"]
    pd.DataFrame(hist).to_csv(os.path.join(RES, f"history_{tag}.csv"), index=False)
    np.save(os.path.join(RES, f"testprob_{tag}.npy"), pt); np.save(os.path.join(RES, "test_labels.npy"), yt)
    pd.DataFrame(rep).T.to_csv(os.path.join(RES, f"perclass_{tag}.csv"))
    pd.DataFrame(confusion_matrix(yt, yp), index=CLASSES, columns=CLASSES).to_csv(os.path.join(RES, f"cm_{tag}.csv"))
    if seed in CFG["SAVE_WEIGHTS_SEEDS"] and exp == CFG["PROPOSED"]:
        shutil.copy(ckpt, os.path.join(RES, f"best_{tag}.pt"))
    os.remove(ckpt)
    runs_csv = os.path.join(RES, "runs.csv")
    old = pd.read_csv(runs_csv) if os.path.exists(runs_csv) else pd.DataFrame()
    pd.concat([old, pd.DataFrame([out])], ignore_index=True).to_csv(runs_csv, index=False)
    print(f"DONE {tag}: acc {out['acc']:.4f} macro-F1 {out['macro_f1']:.4f} | glaze {out['f1_glaze']:.3f} "
          f"frost {out['f1_frost']:.3f} snow {out['f1_snow']:.3f} ({train_min:.1f} min)")
    del m, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return out

In [ ]:
# 6. Run the experiments (skips finished (exp, seed) pairs; stops starting new runs after MAX_HOURS)
runs_csv = os.path.join(RES, "runs.csv"); stopped = False
for seed in CFG["SEEDS"]:
    for exp in CFG["RUN"]:
        done = pd.read_csv(runs_csv) if os.path.exists(runs_csv) else pd.DataFrame(columns=["exp", "seed"])
        if ((done.exp == exp) & (done.seed == seed)).any():
            print("skip", exp, seed); continue
        if (time.time() - T_START) / 3600 + 0.4 > CFG["MAX_HOURS"]:
            print("TIME BUDGET reached, not starting", exp, seed); stopped = True; break
        print(f"\n=== {exp} {EXPS[exp]} | seed {seed} ===")
        run(exp, seed)
    if stopped: break
print(f"total time {(time.time() - T_START) / 3600:.2f} h")

In [ ]:
# 7. Ablation table: mean ± std over seeds, with the ice-hazard classes (glaze, frost, snow) shown separately
r = pd.read_csv(runs_csv)
cols = ["acc", "macro_f1", "mcc", "macro_f1_tta", "f1_glaze", "f1_frost", "f1_snow"]
agg = r.groupby("exp").agg({**{c: ["mean", "std"] for c in cols}, "params_M": "first", "gflops": "first", "seed": "count"})
table = pd.DataFrame({c: (agg[(c, "mean")] * 100).round(2).astype(str) + " ± " + (agg[(c, "std")].fillna(0) * 100).round(2).astype(str)
                      for c in cols})
table["ice_mean_f1"] = (r.groupby("exp")[["f1_glaze", "f1_frost", "f1_snow"]].mean().mean(axis=1) * 100).round(2)
table["params (M)"] = agg[("params_M", "first")]; table["GFLOPs"] = agg[("gflops", "first")]; table["seeds"] = agg[("seed", "count")]
table = table.loc[[e for e in CFG["RUN"] if e in table.index]]
table.to_csv(os.path.join(RES, "ablation_table.csv")); table

In [ ]:
# 8. Per-class F1 of every experiment (mean over seeds)
pc = (r.groupby("exp")[[f"f1_{c}" for c in CLASSES]].mean() * 100).round(1); pc.columns = CLASSES
pc.to_csv(os.path.join(RES, "perclass_f1_mean.csv")); pc

In [ ]:
# 9. McNemar: proposed vs every other experiment, pooled over the seeds that both have
from scipy.stats import chi2
yt = np.load(os.path.join(RES, "test_labels.npy")); P = CFG["PROPOSED"]; rows = []
for o in [e for e in table.index if e != P]:
    b = c = 0
    for s in sorted(set(r[r.exp == P].seed) & set(r[r.exp == o].seed)):
        a = np.load(os.path.join(RES, f"testprob_{P}_s{s}.npy")).argmax(1) == yt
        q = np.load(os.path.join(RES, f"testprob_{o}_s{s}.npy")).argmax(1) == yt
        b += int((a & ~q).sum()); c += int((~a & q).sum())
    st = (abs(b - c) - 1) ** 2 / (b + c) if b + c else 0.0
    rows.append(dict(proposed=P, other=o, proposed_only_right=b, other_only_right=c, chi2=round(st, 3), p_value=chi2.sf(st, 1)))
mc = pd.DataFrame(rows); mc.to_csv(os.path.join(RES, "mcnemar_proposed.csv"), index=False); mc

In [ ]:
# 10. Small zip of results (no weights) for download from the Output tab
import zipfile
zp = os.path.join(WORK, "results_c3r2_seeds" + "".join(str(x) for x in CFG["SEEDS"]) + "_small.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(RES):
        if not f.endswith(".pt"): z.write(os.path.join(RES, f), f)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 1e6))